# 🤖 AutoML Pilot - Training Template
Use this Colab notebook to run advanced AutoML (PyCaret) on your dataset. This template is designed to work seamlessly with datasets exported from the AutoML Pilot web app.

In [ ]:
# @title 1. Install Dependencies
# Install PyCaret and other necessary libraries
!pip install -q pycaret[full] ydata-profiling pandas
print('✅ Dependencies installed!')

In [ ]:
# @title 2. Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
print('✅ Drive mounted! You can now access your files at /content/drive/MyDrive/')

In [ ]:
# @title 3. Load Dataset
import pandas as pd
from pathlib import Path

# EDIT THIS PATH to point to your dataset
dataset_path = '/content/drive/MyDrive/dataset.csv' # @param {type:"string"}

if os.path.exists(dataset_path):
    df = pd.read_csv(dataset_path)
    print(f'✅ Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns')
    display(df.head())
else:
    print(f'❌ Error: File not found at {dataset_path}')

In [ ]:
# @title 4. AutoML Setup & Training
from pycaret.classification import setup as cls_setup, compare_models as cls_compare, pull as cls_pull, save_model as cls_save, finalize_model as cls_finalize
from pycaret.regression import setup as reg_setup, compare_models as reg_compare, pull as reg_pull, save_model as reg_save, finalize_model as reg_finalize
import json

# Configuration
target_column = 'target' # @param {type:"string"}
task_type = 'classification' # @param ["classification", "regression"]

print(f'🚀 Starting AutoML for {task_type} on target: {target_column}')

results_summary = {}

if task_type == 'classification':
    s = cls_setup(data=df, target=target_column, session_id=123, verbose=False)
    print('Comparing models...')
    best_model = cls_compare()
    leaderboard = cls_pull()
    display(leaderboard)
    
    # Finalize and Save
    final_model = cls_finalize(best_model)
    cls_save(final_model, 'best_classification_model')
    print('✅ Classification model saved as best_classification_model.pkl')
    
    best_row = leaderboard.iloc[0]
    results_summary = {
        "model": str(best_model).split('(')[0],
        "task": "Classification",
        "accuracy": float(best_row['Accuracy']),
        "f1_score": float(best_row['F1'])
    }

else:
    s = reg_setup(data=df, target=target_column, session_id=123, verbose=False)
    print('Comparing models...')
    best_model = reg_compare()
    leaderboard = reg_pull()
    display(leaderboard)
    
    # Finalize and Save
    final_model = reg_finalize(best_model)
    reg_save(final_model, 'best_regression_model')
    print('✅ Regression model saved as best_regression_model.pkl')
    
    best_row = leaderboard.iloc[0]
    results_summary = {
        "model": str(best_model).split('(')[0],
        "task": "Regression",
        "r2_score": float(best_row['R2']),
        "rmse": float(best_row['RMSE'])
    }


In [ ]:
# @title 5. Download Model
from google.colab import files

model_name = 'best_classification_model.pkl' if task_type == 'classification' else 'best_regression_model.pkl'
if os.path.exists(model_name):
    files.download(model_name)
    print(f'✅ Triggered download for {model_name}')
else:
    # PyCaret might save without .pkl extension in some versions for the function call
    alt_name = model_name.replace('.pkl', '')
    if os.path.exists(alt_name + '.pkl'):
        files.download(alt_name + '.pkl')
        print(f'✅ Triggered download for {alt_name}.pkl')
    else:
        print('❌ Model file not found.')

In [ ]:
# @title 6. Email Results (Optional)
import smtplib
import ssl
import json
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

RECIPIENT_EMAIL = "" # @param {type:"string"}
SENDER_EMAIL = "" # @param {type:"string"}
APP_PASSWORD = "" # @param {type:"string"}

if RECIPIENT_EMAIL and SENDER_EMAIL and APP_PASSWORD:
    msg = MIMEMultipart("alternative")
    msg["Subject"] = f"AutoML Pilot Colab Results - {results_summary.get('model')}"
    msg["From"] = f"AutoMLPilot Colab <{SENDER_EMAIL}>"
    msg["To"] = RECIPIENT_EMAIL

    html_body = f"""
    <html>
    <body style='font-family:sans-serif; padding:20px;'>
        <h2>🚀 AutoML Pilot Colab Report</h2>
        <div style='background:#f9fafb; padding:15px; border-radius:8px;'>
            <p><strong>Model:</strong> {results_summary.get('model')}</p>
            <p><strong>Task:</strong> {results_summary.get('task')}</p>
            <p><strong>Metric 1:</strong> {results_summary.get('accuracy') if task_type == 'classification' else results_summary.get('r2_score')}</p>
            <p><strong>Metric 2:</strong> {results_summary.get('f1_score') if task_type == 'classification' else results_summary.get('rmse')}</p>
        </div>
        <br>
        <pre>{json.dumps(results_summary, indent=2)}</pre>
    </body>
    </html>
    """
    msg.attach(MIMEText(html_body, "html"))
    
    try:
        context = ssl.create_default_context()
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=context) as server:
            server.login(SENDER_EMAIL, APP_PASSWORD)
            server.sendmail(SENDER_EMAIL, RECIPIENT_EMAIL, msg.as_string())
        print("✅ Email sent successfully!")
    except Exception as e:
        print(f"❌ Failed to send email: {e}")
else:
    print("ℹ️ Email configuration missing. Skipping email step.")